# Phase 7 — Fraud Detection (Python)

**Project:** Banking Transaction Monitoring & Fraud Analytics Platform
**Database:** `federal_bank`

This notebook applies **rule-based** fraud detection in Python, writes the results to
the `fraud_alerts` table, and — importantly — **scores** the results against the
`fraud_ground_truth.csv` we saved in Phase 4, so we can report a real detection rate.

It mirrors the five rules in `fraud_rules.sql` (velocity, amount anomaly, impossible
travel, midnight high-value, card testing). Rules are explainable: every alert records
exactly why it fired.

## 1. Configuration & thresholds

In [ ]:
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "YOUR_MYSQL_PASSWORD",   # <-- your MySQL password
    "database": "federal_bank",
}
GROUND_TRUTH_CSV = "datasets/fraud_ground_truth.csv"

# Rule thresholds (tunable — tighter = fewer false negatives but more false positives)
VELOCITY_WINDOW_SEC = 120   # 2 minutes
VELOCITY_THRESHOLD  = 5     # more than this many transactions in the window
AMOUNT_FACTOR       = 8     # amount above this multiple of the account average
TRAVEL_MAX_MINUTES  = 60    # city change within this many minutes is "impossible"
MIDNIGHT_MAX_HOUR   = 4     # hours 00:00-04:59 count as "midnight"
MIDNIGHT_FACTOR     = 5     # midnight amount above this multiple of account average
CARD_TEST_WINDOW_SEC = 300  # 5 minutes
CARD_TEST_MIN_FAILS  = 3    # this many failures before a success

## 2. Imports & connect

In [ ]:
import pandas as pd
import numpy as np
import mysql.connector

conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()
print("Connected to database:", DB_CONFIG["database"])

## 3. Load transactions into pandas
We join to `accounts` to get each transaction's `customer_id` (needed for alerts).

In [ ]:
query = """
    SELECT t.transaction_id, t.account_id, a.customer_id, t.amount, t.status,
           t.transaction_city, t.transaction_timestamp
    FROM transactions t
    JOIN accounts a ON t.account_id = a.account_id
"""
cursor.execute(query)
cols = [d[0] for d in cursor.description]
df = pd.DataFrame(cursor.fetchall(), columns=cols)
df["amount"] = df["amount"].astype(float)
df["ts"] = pd.to_datetime(df["transaction_timestamp"])
df = df.sort_values(["account_id", "ts"]).reset_index(drop=True)
print(f"Loaded {len(df)} transactions for analysis.")

## 4. Detection rules, alert builder, and scorer
Each rule returns a dict of `{transaction_id: reason}`.

In [ ]:
# Maps each rule to its fraud_alerts ENUM value and a risk level.
RULE_META = {
    "velocity":            ("Velocity",            "Medium"),
    "amount_anomaly":      ("Amount Anomaly",      "High"),
    "impossible_travel":   ("Impossible Travel",   "High"),
    "midnight_high_value": ("Midnight High Value", "High"),
    "card_testing":        ("Card Testing",        "Medium"),
}


def flag_velocity(df, window_sec, threshold):
    # Flag any transaction that is the (threshold+1)-th or later within a trailing window.
    flagged = {}
    for aid, g in df.groupby("account_id"):
        times = g["ts"].values.astype("datetime64[s]").astype("int64")
        tids = g["transaction_id"].values
        for i in range(len(times)):
            lo = np.searchsorted(times, times[i] - window_sec, side="left")
            count = i - lo + 1
            if count > threshold:
                flagged[int(tids[i])] = f"{count} transactions within {window_sec // 60} min"
    return flagged


def flag_amount_anomaly(df, factor):
    avg = df.groupby("account_id")["amount"].transform("mean")
    mask = df["amount"] > factor * avg
    return {int(t): f"Amount {a:.2f} exceeds {factor}x account average {m:.2f}"
            for t, a, m in zip(df.loc[mask, "transaction_id"], df.loc[mask, "amount"], avg[mask])}


def flag_impossible_travel(df, max_minutes):
    flagged = {}
    for aid, g in df.groupby("account_id"):
        g = g.reset_index(drop=True)
        for i in range(1, len(g)):
            c0, c1 = g.loc[i - 1, "transaction_city"], g.loc[i, "transaction_city"]
            if c0 and c1 and c0 != c1:
                mins = (g.loc[i, "ts"] - g.loc[i - 1, "ts"]).total_seconds() / 60
                if 0 <= mins <= max_minutes:
                    flagged[int(g.loc[i, "transaction_id"])] = f"Moved {c0} -> {c1} in {mins:.0f} min"
    return flagged


def flag_midnight(df, factor, max_hour):
    avg = df.groupby("account_id")["amount"].transform("mean")
    hour = df["ts"].dt.hour
    mask = (hour <= max_hour) & (df["amount"] > factor * avg)
    return {int(t): f"High-value {a:.2f} at {h:02d}:00 hrs"
            for t, a, h in zip(df.loc[mask, "transaction_id"], df.loc[mask, "amount"], hour[mask])}


def flag_card_testing(df, window_sec, min_fails):
    flagged = {}
    for aid, g in df.groupby("account_id"):
        times = g["ts"].values.astype("datetime64[s]").astype("int64")
        status = g["status"].values
        tids = g["transaction_id"].values
        for i in range(len(times)):
            if status[i] == "Success":
                lo = np.searchsorted(times, times[i] - window_sec, side="left")
                fails = sum(1 for j in range(lo, i) if status[j] == "Failed")
                if fails >= min_fails:
                    flagged[int(tids[i])] = f"{fails} failed attempts before this success"
    return flagged


def build_alerts(df, flags_by_rule):
    # Turn {rule: {tid: reason}} into a list of alert rows.
    info = df.set_index("transaction_id")[["account_id", "customer_id", "ts"]].to_dict("index")
    alerts = []
    for rule, (enum_name, risk) in RULE_META.items():
        for tid, reason in flags_by_rule.get(rule, {}).items():
            meta = info[tid]
            alerts.append({
                "transaction_id": int(tid),
                "account_id": int(meta["account_id"]),          # kept for scoring (not stored)
                "customer_id": int(meta["customer_id"]),
                "rule_triggered": enum_name,
                "alert_reason": reason,
                "risk_level": risk,
                "alert_timestamp": meta["ts"].to_pydatetime(),
            })
    return alerts


def score_detection(alerts_df, ground_truth_df):
    # For each seeded fraud (account_id + fraud_type), did we raise a matching alert?
    enum_to_type = {v[0]: k for k, v in RULE_META.items()}
    a = alerts_df.copy()
    a["fraud_type"] = a["rule_triggered"].map(enum_to_type)
    detected = set(zip(a["account_id"], a["fraud_type"]))

    instances = (ground_truth_df.groupby(["account_id", "fraud_type"])
                 .size().reset_index()[["account_id", "fraud_type"]])
    rows = []
    for ftype, grp in instances.groupby("fraud_type"):
        seeded = len(grp)
        caught = sum((int(r.account_id), r.fraud_type) in detected for r in grp.itertuples())
        rows.append({"fraud_type": ftype, "seeded": seeded, "detected": caught,
                     "detection_rate": f"{(caught / seeded * 100):.0f}%"})
    return pd.DataFrame(rows).sort_values("fraud_type").reset_index(drop=True)

## 5. Run detection

In [ ]:
flags_by_rule = {
    "velocity":            flag_velocity(df, VELOCITY_WINDOW_SEC, VELOCITY_THRESHOLD),
    "amount_anomaly":      flag_amount_anomaly(df, AMOUNT_FACTOR),
    "impossible_travel":   flag_impossible_travel(df, TRAVEL_MAX_MINUTES),
    "midnight_high_value": flag_midnight(df, MIDNIGHT_FACTOR, MIDNIGHT_MAX_HOUR),
    "card_testing":        flag_card_testing(df, CARD_TEST_WINDOW_SEC, CARD_TEST_MIN_FAILS),
}
for rule, hits in flags_by_rule.items():
    print(f"{rule:<20}: {len(hits)} transactions flagged")

alerts = build_alerts(df, flags_by_rule)
alerts_df = pd.DataFrame(alerts)
print(f"\nTotal alerts: {len(alerts_df)}")

## 6. Write alerts to the fraud_alerts table
We clear old alerts, mark the flagged transactions, and insert the new alerts.

In [ ]:
cursor.execute("TRUNCATE TABLE fraud_alerts")

# Mark the underlying transactions as flagged (useful for the dashboard).
flagged_ids = tuple(sorted({a["transaction_id"] for a in alerts}))
cursor.execute("UPDATE transactions SET is_flagged = FALSE")   # reset first
if flagged_ids:
    fmt = ", ".join(["%s"] * len(flagged_ids))
    cursor.execute(f"UPDATE transactions SET is_flagged = TRUE WHERE transaction_id IN ({fmt})", flagged_ids)

INSERT_ALERT = (
    "INSERT INTO fraud_alerts "
    "(transaction_id, customer_id, rule_triggered, alert_reason, risk_level, status, alert_timestamp) "
    "VALUES (%s, %s, %s, %s, %s, 'Open', %s)"
)
alert_tuples = [
    (a["transaction_id"], a["customer_id"], a["rule_triggered"],
     a["alert_reason"], a["risk_level"], a["alert_timestamp"])
    for a in alerts
]
if alert_tuples:
    cursor.executemany(INSERT_ALERT, alert_tuples)
conn.commit()
print(f"Wrote {len(alert_tuples)} alerts to fraud_alerts and flagged {len(flagged_ids)} transactions.")

## 7. Score against the ground truth
How many of the fraud cases we *seeded* in Phase 4 did the rules actually catch?

In [ ]:
ground_truth = pd.read_csv(GROUND_TRUTH_CSV)
score = score_detection(alerts_df, ground_truth)
print("Detection rate by fraud type:\n")
print(score.to_string(index=False))

# How many alerts correspond to a seeded fraud vs. are additional flags?
enum_to_type = {v[0]: k for k, v in RULE_META.items()}
alerts_df["fraud_type"] = alerts_df["rule_triggered"].map(enum_to_type)
seeded_pairs = set(zip(ground_truth["account_id"], ground_truth["fraud_type"]))
alerts_df["matches_seeded"] = [
    (r.account_id, r.fraud_type) in seeded_pairs for r in alerts_df.itertuples()
]
print(f"\nAlerts matching a seeded fraud: {alerts_df['matches_seeded'].sum()} "
      f"/ {len(alerts_df)} total")
print("(Alerts that don't match a seeded case are additional flags — often genuine "
      "overlapping patterns in the data, and a useful discussion point on false positives.)")

## 8. Peek at some alerts

In [ ]:
cursor.execute("""
    SELECT rule_triggered, COUNT(*) FROM fraud_alerts GROUP BY rule_triggered
""")
print("Alerts in fraud_alerts by rule:")
for rule, count in cursor.fetchall():
    print(f"  {rule:<22}: {count}")

cursor.execute("""
    SELECT alert_id, transaction_id, rule_triggered, risk_level, alert_reason
    FROM fraud_alerts ORDER BY alert_id LIMIT 8
""")
print("\nSample alerts:")
for row in cursor.fetchall():
    print(f"  #{row[0]} txn {row[1]} | {row[2]} ({row[3]}) — {row[4]}")

## 9. Close

In [ ]:
cursor.close()
conn.close()
print("Fraud detection complete. Alerts are ready for the KPIs and dashboard.")